# REST-RECT setup

Automated setup of REST-RECT simulations for glycan systems.

The notebook is intentionally a **control panel**: glycan-specific information is stored in `config/glycans.yaml`, while the reusable workflow is implemented in `scripts/setup.py`.

In [1]:
from pathlib import Path
import sys
import pandas as pd

# Run this notebook from the repository root or from notebooks/.
REPO_ROOT = Path.cwd()
if REPO_ROOT.name == 'notebooks':
    REPO_ROOT = REPO_ROOT.parent

sys.path.insert(0, str(REPO_ROOT / 'scripts'))

from setup import (
    load_config,
    available_glycans,
    overview_dataframe,
    validate_config,
    preview_setup,
    setup_rest_rect,
    add_glycan_to_yaml,
    update_glycan_in_yaml
)

CONFIG_FILE = REPO_ROOT / 'config' / 'glycans.yaml'
TEMPLATE_DIR = REPO_ROOT / 'templates'
config = load_config(CONFIG_FILE)

print(f'Repository: {REPO_ROOT}')
print(f'Configuration: {CONFIG_FILE}')

Repository: /work/grothaus/GlycoSOFTWARE/GlycoGEN
Configuration: /work/grothaus/GlycoSOFTWARE/GlycoGEN/config/glycans.yaml


## 1. Check the configuration

This checks linkage lengths, residue labels, replica count and temperature ranges before any files or simulations are touched.

In [4]:
problems = validate_config(config)

if problems:
    for glycan, errors in problems.items():
        print(f'❌ {glycan}')
        for error in errors:
            print(f'   - {error}')
    raise ValueError('Configuration contains errors.')
else:
    print(f'✓ Configuration valid: {len(available_glycans(config))} glycans')

✓ Configuration valid: 32 glycans


## 2. Glycan overview

In [5]:
overview_dataframe(config)

,glycan,path,replicas,T_min [K],T_max [K],peptide,residues,linkages,monosaccharide types
0,Man5,../M/Man5/,12,310.15,800.0,ANA,7,6,"BGLCNA, AGALNA, ANE5AC, AFUC, BGAL, BMAN, AMAN"
1,Man5_Fuc1,../M/Man5_Fuc1/,12,310.15,800.0,ANA,8,7,"BGLCNA, AGALNA, ANE5AC, AFUC, BGAL, BMAN, AMAN"
2,Man5_GlcNAc1,../M/Man5_GlcNAc1/,12,310.15,800.0,ANA,8,7,"BGLCNA, AGALNA, ANE5AC, AFUC, BGAL, BMAN, AMAN"
3,Man6,../M/Man6/,12,310.15,800.0,ANA,8,7,"BGLCNA, AGALNA, ANE5AC, AFUC, BGAL, BMAN, AMAN"
4,Man7,../M/Man7/,12,310.15,800.0,ANA,9,8,"BGLCNA, AGALNA, ANE5AC, AFUC, BGAL, BMAN, AMAN"
5,Man8,../M/Man8/,12,310.15,800.0,ANA,10,9,"BGLCNA, AGALNA, ANE5AC, AFUC, BGAL, BMAN, AMAN"
6,Man9,data/N-glycans/High-mannose/Man9/,12,310.15,800.0,ANA,11,10,"BGLCNA, AGALNA, ANE5AC, AFUC, BGAL, BMAN, AMAN"
7,Man9_Glc3,../M/Man9_Glc3/,12,310.15,800.0,ANA,14,13,"BGLCNA, AGALNA, ANE5AC, AFUC, BGAL, BMAN, AMAN..."
8,Man9_Glc1,../M/Man9_Glc1/,12,310.15,800.0,ANA,12,11,"BGLCNA, AGALNA, ANE5AC, AFUC, BGAL, BMAN, AMAN..."
9,A1F,../C/A1F/,12,310.15,800.0,ANA,11,10,"BGLCNA, AGALNA, ANE5AC, AFUC, BGAL, BMAN, AMAN"


## 3. Add/Modify a glycan manually

Use this helper when adding a glycan that is not yet in `config/glycans.yaml`. Only parameters that differ from the global defaults need to be supplied. The function validates the definition before writing it to the YAML file.

**Tip:** keep the residue/linkage order exactly as used by the GlyCONFORMER labelling scheme.


In [ ]:
 add_glycan_to_yaml(
     CONFIG_FILE,
     name='A2G2S2',
     path='../C/A2G2S2/',
     label=[[1, 2], [2, 3], [3, 4]],
     ltype=[4, 4, 6],
     carbon_t=[1, 1, 1],
     carbon_p=[1, 1, 1, 1],
 )


In [9]:
update_glycan_in_yaml(
    CONFIG_FILE,
    name="Man9",
    path="data/N-glycans/High-mannose/Man9/",
)

Updated glycan 'Man9' in /work/grothaus/GlycoSOFTWARE/GlycoGEN/config/glycans.yaml


{'path': 'data/N-glycans/High-mannose/Man9/',
 'label': [[1, 2],
  [2, 3],
  [3, 4],
  [4, 5],
  [5, 6],
  [4, 7],
  [7, 8],
  [3, 9],
  [9, 10],
  [10, 11]],
 'ltype': [4, 4, 6, 6, 2, 3, 2, 3, 2, 2],
 'carbon_t': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1],
 'carbon_p': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}

## 4. Select glycans
Examples:
```python
selected_glycans = ['Man9']
selected_glycans = ['Man5', 'Man6', 'Man7', 'Man8', 'Man9']
selected_glycans = available_glycans(config)
```

In [15]:
selected_glycans = ['Man9']

unknown = sorted(set(selected_glycans) - set(available_glycans(config)))
if unknown:
    raise KeyError(f'Unknown glycans: {unknown}')

print('Selected:', ', '.join(selected_glycans))

Selected: Man9


## 5. Preview before execution

This does not execute GROMACS or modify simulation directories. It is deliberately separate from the execution cell.

In [2]:
selected_glycans = ['Man9']

for glycan in selected_glycans:
    preview_setup(glycan, config, repo_root=REPO_ROOT)

REST-RECT setup: Man9
Simulation directory : /work/grothaus/GlycoSOFTWARE/GlycoGEN/data/N-glycans/High-mannose/Man9
Replicas             : 12
Temperature          : 310.15 -> 800.0 K
Structure            : step5_production.gro
REST-RECT MDP        : production.mdp
Peptide              : ANA
Residues             : 11
Linkages             : 10
Monosaccharides      : BGLCNA, AGALNA, ANE5AC, AFUC, BGAL, BMAN, AMAN
Linkages:
  1-2 (type 4), 2-3 (type 4), 3-4 (type 6), 4-5 (type 6), 5-6 (type 2), 4-7 (type 3), 7-8 (type 2), 3-9 (type 3), 9-10 (type 2), 10-11 (type 2)


## 5. Execute

### Cluster settings

For the cluster used by the original notebook, set `USE_MODULE_LOAD = True`. This uses `ml chem/GROMACS` before GROMACS commands. On a system where `gmx` and `gmx_mpi` are already on `PATH`, leave it `False`.

`FORCE = False` is intentionally conservative: existing simulation directories are not overwritten automatically.

In [3]:
USE_MODULE_LOAD = True
FORCE = False

for glycan in selected_glycans:
    setup_rest_rect(
        glycan,
        config,
        repo_root=REPO_ROOT,
        template_dir=TEMPLATE_DIR,
        use_module_load=USE_MODULE_LOAD,
        force=FORCE,
        dry_run=False
    )


Glycan: Man9
Simulation directory: /work/grothaus/GlycoSOFTWARE/GlycoGEN/data/N-glycans/High-mannose/Man9
✓ Glycan directory exists
✓ CHARMM-GUI directory detected
✓ Minimisation/equilibration already performed:
✗ REST-RECT simulation not yet setup:
plumed_allangles.dat does not exist in the directory.
reference.pdb does not exist in the directory.
processed.top does not exist in the directory.
processed_m.top does not exist in the directory.
$ bash -lc ml chem/GROMACS && gmx editconf -f 01_min_equ/step5_production.gro -o reference.pdb -resnr 1
              :-) GROMACS - gmx editconf, 2022.6-plumed_2.8.3 (-:

Executable:   /home/applications/software/EB-SOFTWARE-BASE/software/GROMACS/2022.6-foss-2022a-CUDA-11.7.0-PLUMED-2.8.3/bin/gmx
Data prefix:  /home/applications/software/EB-SOFTWARE-BASE/software/GROMACS/2022.6-foss-2022a-CUDA-11.7.0-PLUMED-2.8.3
Working dir:  /work/grothaus/GlycoSOFTWARE/GlycoGEN/data/N-glycans/High-mannose/Man9
Command line:
  gmx editconf -f 01_min_equ/step5_p

## Notes

- `metadata.yaml` is written into each simulation directory after a successful setup.
- `rest_rect_setup.log` records commands and output.
- The YAML contains the glycan topology information in the same order as the original notebook. Do not reorder `label` entries unless the GlyCONFORMER labelling scheme is changed accordingly.
- Add further glycans to `config/glycans.yaml`; no notebook changes are needed.